In [1]:
import sys
from pathlib import Path

# Asumsikan kamu menjalankan notebook dari 'data_pipeline_pyspark/notebooks'
# dan kamu ingin import dari 'data_pipeline_pyspark/src'
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd

from src.utils.helper import startup_investments_engine_pyspark

from src.staging.extract.extract_db import extract_database
from src.staging.extract.extract_db_pyspark import extract_database as extract_database_pyspark
from src.staging.extract.extract_spreadsheet_pyspark import extract_sheet_spark,extract_spreadsheet as extract_spreadsheet_pyspark

from src.staging.extract.extract_spreadsheet import extract_spreadsheet
from src.staging.extract.extract_api import extract_api_milestones
from src.staging.load.load import load_staging
from src.staging.extract.extract_spreadsheet import extract_spreadsheet, extract_sheet

from src.warehouse.extract.extract_db import extract_database as extract_staging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from src.staging.load.load_pyspark import load_staging_pyspark_upsert

# Declare spark

In [2]:
# create spark session
spark = SparkSession.builder \
    .appName("Pipeline Staging") \
    .config("spark.ui.enabled", "true") \
    .getOrCreate()

print(spark.version)  # Menampilkan versi Spark


3.3.2


In [11]:
acquisition = extract_database_pyspark(spark=spark,table_name='acquisition')
acquisition

+-------+----------+-------+--------+-----------+-------------------+---------+
|   step|   process| status|  source| table_name|           etl_date|error_msg|
+-------+----------+-------+--------+-----------+-------------------+---------+
|staging|extraction|success|database|acquisition|2025-04-22 15:03:32|     null|
+-------+----------+-------+--------+-----------+-------------------+---------+



DataFrame[acquisition_id: int, acquiring_object_id: string, acquired_object_id: string, term_code: string, price_amount: decimal(15,2), price_currency_code: string, acquired_at: timestamp, source_url: string, source_description: string, created_at: timestamp, updated_at: timestamp]

In [12]:
acquisition.show()

+--------------+-------------------+------------------+---------+------------+-------------------+-----------+----------+------------------+----------+----------+
|acquisition_id|acquiring_object_id|acquired_object_id|term_code|price_amount|price_currency_code|acquired_at|source_url|source_description|created_at|updated_at|
+--------------+-------------------+------------------+---------+------------+-------------------+-----------+----------+------------------+----------+----------+
+--------------+-------------------+------------------+---------+------------+-------------------+-----------+----------+------------------+----------+----------+



In [13]:
load_staging_pyspark_upsert(spark, data=acquisition, schema='public', table_name='acquisition', idx_name='acquisition_id', source='database')


+-------+-------+-------+--------+-----------+--------------------+---------+
|   step|process| status|  source| table_name|            etl_date|error_msg|
+-------+-------+-------+--------+-----------+--------------------+---------+
|staging|   load|success|database|acquisition|2025-04-22 15:04:...|     null|
+-------+-------+-------+--------+-----------+--------------------+---------+



In [3]:
# relationship 
people_df = extract_spreadsheet_pyspark(spark=spark,table_name='people')
people_df

# # people 
# relationships_df = extract_spreadsheet(table_name='relationships')
# relationships_df


+-------+-----------+-------+--------+----------+-------------------+---------+
|   step|    process| status|  source|table_name|           etl_date|error_msg|
+-------+-----------+-------+--------+----------+-------------------+---------+
|staging|spreadsheet|success|database|    people|2025-04-22 16:15:40|     null|
+-------+-----------+-------+--------+----------+-------------------+---------+



DataFrame[people_id: string, object_id: string, first_name: string, last_name: string, birthplace: string, affiliation_name: string, created_at: string]

In [4]:
people_df.show()

+---------+---------+----------+----------+--------------------+--------------------+-------------------+
|people_id|object_id|first_name| last_name|          birthplace|    affiliation_name|         created_at|
+---------+---------+----------+----------+--------------------+--------------------+-------------------+
|        1|      p:2|       Ben|   Elowitz|                null|           Blue Nile|2025-04-22 16:15:40|
|        2|      p:3|     Kevin|  Flaherty|                null|            Wetpaint|2025-04-22 16:15:40|
|        3|      p:4|      Raju|   Vegesna|                null|                Zoho|2025-04-22 16:15:40|
|        4|      p:5|       Ian|     Wenig|                null|                Zoho|2025-04-22 16:15:40|
|        5|      p:6|     Kevin|      Rose|         Redding, CA|        i/o Ventures|2025-04-22 16:15:40|
|        6|      p:7|       Jay|   Adelson|         Detroit, MI|                Digg|2025-04-22 16:15:40|
|        7|      p:8|      Owen|     Byrne|   

# Staging

## Extract

### api

In [ ]:
df_staging_api = extract_api_milestones(table_name='milestones')
df_staging_api

,created_at,description,milestone_at,milestone_code,milestone_id,object_id,source_description,source_url,updated_at
0,2009-05-24 10:42:44,"February 1, 1873: Mirror Printing Office and B...",1960-01-01,other,1314,c:2438,Los Angeles Times Media CEnter,http://www.latimes.com/services/newspaper/medi...,2009-05-25 15:56:17.000
1,2010-11-17 13:51:42,TAB celebrates 10th anniversary of founding by...,1960-01-01,other,8545,c:61426,History of TAB,http://www.tab.com/About/Media/History.aspx,2010-11-17 19:17:30.000
2,2012-06-20 03:44:44,Starting the ongoing tradition of maintaining ...,1960-01-01,other,20935,c:152373,Widen History Timeline,NaN,2012-06-21 09:34:49.000
0,2013-12-03 00:02:57,CREDITANSTALT - LWA Architect firm in Offenbac...,1961-03-01,other,38963,c:35045,"Architect, Architektenkammer Frankfurt am Main...",http://related.ca.ag,2013-12-03 09:27:56.000
0,2011-03-30 04:27:32,head of the office,1963-01-01,other,11861,c:59346,NaN,NaN,2011-04-02 00:36:39.000
...,...,...,...,...,...,...,...,...,...
28,2013-11-11 18:40:45,Semmy Rülf announced as new Chairman of the Board,2014-11-01,other,37923,c:175766,Semmy Rülf new chairman at Incentive - talks a...,http://oresundstartups.com/semmy-rulf-new-chai...,2013-11-12 00:43:17.000
29,2013-11-12 12:01:12,Swiss Mobility Solutions today confirmed its p...,2014-02-24,other,37948,c:235635,NaN,NaN,2013-11-12 19:44:09.000
30,2013-11-15 17:46:15,Sigma Alimentos has offered to acquire Campofr...,2014-11-15,other,38170,c:279873,NaN,http://www.campofriofoodgroup.com,2013-11-15 17:46:15.000
31,2013-11-18 07:11:10,Rebranded from Healthse.in to WorkoutTrends.com,2014-11-15,other,38236,c:255923,Voyage from Healthse.in to WorkoutTrends.com,http://workouttrends.com/whyd-we-rebrand,2013-11-18 10:45:45.000


### spreadsheet

In [ ]:
# relationship 
people_df = extract_spreadsheet(table_name='people')
people_df

# people 
relationships_df = extract_spreadsheet(table_name='relationships')
relationships_df


,relationship_id,person_object_id,relationship_object_id,start_at,end_at,is_past,sequence,title,created_at,updated_at
1,1,p:2,c:1,NaN,NaN,FALSE,8,Co-Founder/CEO/Board of Directors,2007-05-25 07:03:54,2013-06-03 09:58:46.000
2,2,p:3,c:1,NaN,NaN,FALSE,279242,VP Marketing,2007-05-25 07:04:16,2010-05-21 16:31:34.000
3,3,p:4,c:3,NaN,NaN,FALSE,4,Evangelist,2007-05-25 19:33:03,2013-06-29 13:36:58.000
4,4,p:5,c:3,2006-03-01 00:00:00.000,2009-12-01 00:00:00.000,FALSE,4,Senior Director Strategic Alliances,2007-05-25 19:34:53,2013-06-29 10:25:34.000
5,6,p:7,c:4,2005-07-01 00:00:00.000,2010-04-05 00:00:00.000,FALSE,1,Chief Executive Officer,2007-05-25 20:05:33,2010-04-05 18:41:41.000
...,...,...,...,...,...,...,...,...,...,...
134722,196878,p:35873,c:59431,2012-04-01 00:00:00.000,NaN,FALSE,6,Board Member,2013-02-14 17:30:02,2013-08-12 09:07:21.000
134723,196879,p:35873,c:151478,NaN,NaN,FALSE,8,Investor,2013-02-14 17:30:32,2013-08-12 09:07:21.000
134724,196881,p:4345,c:187489,2012-08-01 00:00:00.000,NaN,FALSE,306113,President,2013-02-14 17:33:44,2013-02-14 20:39:26.000
134725,196886,p:137337,c:187489,2012-09-15 00:00:00.000,NaN,FALSE,2,Chief Financial Officer,2013-02-14 17:36:01,2013-02-14 20:39:36.000


### DB

In [ ]:
# acquisition

acquisition = extract_database('acquisition')

#company
company = extract_database('company')

#funding_rounds
funding_rounds = extract_database('funding_rounds')

#funds
funds = extract_database('funds')

#investments
investments = extract_database('investments')

#ipos
ipos = extract_database('ipos')


In [ ]:
ipos

,ipo_id,object_id,valuation_amount,valuation_currency_code,raised_amount,raised_currency_code,public_at,stock_symbol,source_url,source_description,created_at,updated_at


## Load

### spreadsheet

In [ ]:
load_staging(data=people_df, table_name='people', schema='public', idx_name='people_id', source='spreadsheet')
load_staging(data=relationships_df, table_name='relationships', schema='public', idx_name='relationship_id', source='spreadsheet')

### api

In [ ]:
load_staging(data=df_staging_api, table_name='milestones', schema='public', idx_name='milestone_id', source='api')


### DB

In [ ]:
# acquisition
load_staging(data=acquisition, table_name='acquisition', schema='public', idx_name='acquisition_id', source='database')
#company
load_staging(data=company, table_name='company', schema='public', idx_name='object_id', source='database')

#funding_rounds
load_staging(data=funding_rounds, table_name='funding_rounds', schema='public', idx_name='funding_round_id', source='database')

#funds
load_staging(data=funds, table_name='funds', schema='public', idx_name='fund_id', source='database')

#investments
load_staging(data=investments, table_name='investments', schema='public', idx_name='investment_id', source='database')

#ipos
load_staging(data=ipos, table_name='ipos', schema='public', idx_name='ipo_id', source='database')


# Warehouse

## Extract

### api

In [ ]:
milestones_staging = extract_staging(table_name='milestones')
milestones_staging

,milestone_id,created_at,description,milestone_at,milestone_code,object_id,source_description,source_url,updated_at
0,1314,2009-05-24 10:42:44,"February 1, 1873: Mirror Printing Office and B...",1960-01-01,other,c:2438,Los Angeles Times Media CEnter,http://www.latimes.com/services/newspaper/medi...,2009-05-25 15:56:17.000
1,8545,2010-11-17 13:51:42,TAB celebrates 10th anniversary of founding by...,1960-01-01,other,c:61426,History of TAB,http://www.tab.com/About/Media/History.aspx,2010-11-17 19:17:30.000
2,20935,2012-06-20 03:44:44,Starting the ongoing tradition of maintaining ...,1960-01-01,other,c:152373,Widen History Timeline,None,2012-06-21 09:34:49.000
3,38963,2013-12-03 00:02:57,CREDITANSTALT - LWA Architect firm in Offenbac...,1961-03-01,other,c:35045,"Architect, Architektenkammer Frankfurt am Main...",http://related.ca.ag,2013-12-03 09:27:56.000
4,11861,2011-03-30 04:27:32,head of the office,1963-01-01,other,c:59346,None,None,2011-04-02 00:36:39.000
...,...,...,...,...,...,...,...,...,...
28682,37923,2013-11-11 18:40:45,Semmy Rülf announced as new Chairman of the Board,2014-11-01,other,c:175766,Semmy Rülf new chairman at Incentive - talks a...,http://oresundstartups.com/semmy-rulf-new-chai...,2013-11-12 00:43:17.000
28683,37948,2013-11-12 12:01:12,Swiss Mobility Solutions today confirmed its p...,2014-02-24,other,c:235635,None,None,2013-11-12 19:44:09.000
28684,38170,2013-11-15 17:46:15,Sigma Alimentos has offered to acquire Campofr...,2014-11-15,other,c:279873,None,http://www.campofriofoodgroup.com,2013-11-15 17:46:15.000
28685,38236,2013-11-18 07:11:10,Rebranded from Healthse.in to WorkoutTrends.com,2014-11-15,other,c:255923,Voyage from Healthse.in to WorkoutTrends.com,http://workouttrends.com/whyd-we-rebrand,2013-11-18 10:45:45.000


### spreadsheet

In [ ]:
# relationship 
people = extract_staging(table_name='people')
people

# people 
relationships = extract_staging(table_name='relationships')
relationships


,relationship_id,person_object_id,relationship_object_id,start_at,end_at,is_past,sequence,title,created_at,updated_at
0,1,p:2,c:1,None,None,FALSE,8,Co-Founder/CEO/Board of Directors,2007-05-25 07:03:54,2013-06-03 09:58:46.000
1,2,p:3,c:1,None,None,FALSE,279242,VP Marketing,2007-05-25 07:04:16,2010-05-21 16:31:34.000
2,3,p:4,c:3,None,None,FALSE,4,Evangelist,2007-05-25 19:33:03,2013-06-29 13:36:58.000
3,4,p:5,c:3,2006-03-01 00:00:00.000,2009-12-01 00:00:00.000,FALSE,4,Senior Director Strategic Alliances,2007-05-25 19:34:53,2013-06-29 10:25:34.000
4,6,p:7,c:4,2005-07-01 00:00:00.000,2010-04-05 00:00:00.000,FALSE,1,Chief Executive Officer,2007-05-25 20:05:33,2010-04-05 18:41:41.000
...,...,...,...,...,...,...,...,...,...,...
134721,196878,p:35873,c:59431,2012-04-01 00:00:00.000,None,FALSE,6,Board Member,2013-02-14 17:30:02,2013-08-12 09:07:21.000
134722,196879,p:35873,c:151478,None,None,FALSE,8,Investor,2013-02-14 17:30:32,2013-08-12 09:07:21.000
134723,196881,p:4345,c:187489,2012-08-01 00:00:00.000,None,FALSE,306113,President,2013-02-14 17:33:44,2013-02-14 20:39:26.000
134724,196886,p:137337,c:187489,2012-09-15 00:00:00.000,None,FALSE,2,Chief Financial Officer,2013-02-14 17:36:01,2013-02-14 20:39:36.000


### DB

In [ ]:
# acquisition

acquisition = extract_staging('acquisition')

#company
company = extract_staging('company')

#funding_rounds
funding_rounds = extract_staging('funding_rounds')

#funds
funds = extract_staging('funds')

#investments
investments = extract_staging('investments')

#ipos
ipos = extract_staging('ipos')


### transform

In [ ]:
import sys
from pathlib import Path

# Asumsikan kamu menjalankan notebook dari 'data_pipeline_pyspark/notebooks'
# dan kamu ingin import dari 'data_pipeline_pyspark/src'
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd


from src.staging.extract.extract_db import extract_database
from src.staging.extract.extract_spreadsheet import extract_spreadsheet
from src.staging.extract.extract_api import extract_api_milestones
from src.staging.load.load import load_staging
from src.staging.extract.extract_spreadsheet import extract_spreadsheet, extract_sheet

from src.warehouse.extract.extract_db import extract_database as extract_staging

from src.warehouse.transform.dim_company import transform_dim_company
from src.warehouse.transform.dim_people import transform_dim_people
from src.warehouse.transform.dim_relationship import transform_dim_relationship
# from src.warehouse.transform.fact_acquisitions import transform_fact_acquisitions
# from src.warehouse.transform.fact_funding_rounds import transform_fact_funding_rounds
# from src.warehouse.transform.fact_funds import transform_fact_funds
# from src.warehouse.transform.fact_investments import transform_fact_investments
# from src.warehouse.transform.fact_ipos import transform_fact_ipos
# from src.warehouse.transform.fact_milestones import transform_fact_milestones

from src.warehouse.load.load import load_warehouse


In [ ]:
# relationship 
people = extract_staging(table_name='people')
people

# people 
relationships = extract_staging(table_name='relationships')
relationships

# acquisition

acquisition = extract_staging('acquisition')

#company
company = extract_staging('company')

#funding_rounds
funding_rounds = extract_staging('funding_rounds')

#funds
funds = extract_staging('funds')

#investments
investments = extract_staging('investments')

#ipos
ipos = extract_staging('ipos')


In [ ]:
relationships['start_at'] = (
    pd.to_datetime(relationships['start_at'], errors='coerce')
    .fillna(pd.Timestamp('2100-01-01'))
    .dt.date
)
relationships['start_at'] = relationships['start_at'].astype('int')

,relationship_id,person_object_id,relationship_object_id,start_at,end_at,is_past,sequence,title,created_at,updated_at
0,1,p:2,c:1,4102444800000000000,None,FALSE,8,Co-Founder/CEO/Board of Directors,2007-05-25 07:03:54,2013-06-03 09:58:46.000
1,2,p:3,c:1,4102444800000000000,None,FALSE,279242,VP Marketing,2007-05-25 07:04:16,2010-05-21 16:31:34.000
2,3,p:4,c:3,4102444800000000000,None,FALSE,4,Evangelist,2007-05-25 19:33:03,2013-06-29 13:36:58.000
3,4,p:5,c:3,1141171200000000000,2009-12-01 00:00:00.000,FALSE,4,Senior Director Strategic Alliances,2007-05-25 19:34:53,2013-06-29 10:25:34.000
4,6,p:7,c:4,1120176000000000000,2010-04-05 00:00:00.000,FALSE,1,Chief Executive Officer,2007-05-25 20:05:33,2010-04-05 18:41:41.000
...,...,...,...,...,...,...,...,...,...,...
134721,196878,p:35873,c:59431,1333238400000000000,None,FALSE,6,Board Member,2013-02-14 17:30:02,2013-08-12 09:07:21.000
134722,196879,p:35873,c:151478,4102444800000000000,None,FALSE,8,Investor,2013-02-14 17:30:32,2013-08-12 09:07:21.000
134723,196881,p:4345,c:187489,1343779200000000000,None,FALSE,306113,President,2013-02-14 17:33:44,2013-02-14 20:39:26.000
134724,196886,p:137337,c:187489,1347667200000000000,None,FALSE,2,Chief Financial Officer,2013-02-14 17:36:01,2013-02-14 20:39:36.000


In [ ]:

# # dim_company
# dim_company = transform_dim_company(company,'company')

# # dim_company
# dim_people = transform_dim_people(people,'people')

# dim_company
dim_relationships = transform_dim_relationship(relationships,'relationships')
dim_relationships
# # dim_company
# dim_company = transform_dim_company(company,'dim_company')
# dim_people


Converting from datetime64[ns] to int32 is not supported. Do obj.astype('int64').astype(dtype) instead
module 'datetime' has no attribute 'now'


## Load

In [ ]:
load_warehouse(data=dim_company, table_name='dim_company', schema='public', 
               idx_name='company_nk', source='staging',table_process='company')

load_warehouse(data=dim_people, table_name='dim_people', schema='public', 
               idx_name='people_nk', source='staging',table_process='people')



### spreadsheet

In [ ]:
load_staging(data=people_df, table_name='people', schema='public', idx_name='people_id', source='spreadsheet')
load_staging(data=relationships_df, table_name='relationships', schema='public', idx_name='relationship_id', source='spreadsheet')

### api

In [ ]:
load_staging(data=df_staging_api, table_name='milestones', schema='public', idx_name='milestone_id', source='api')


### DB

In [ ]:
# acquisition
load_staging(data=acquisition, table_name='acquisition', schema='public', idx_name='acquisition_id', source='database')
#company
load_staging(data=company, table_name='company', schema='public', idx_name='object_id', source='database')

#funding_rounds
load_staging(data=funding_rounds, table_name='funding_rounds', schema='public', idx_name='funding_round_id', source='database')

#funds
load_staging(data=funds, table_name='funds', schema='public', idx_name='fund_id', source='database')

#investments
load_staging(data=investments, table_name='investments', schema='public', idx_name='investment_id', source='database')

#ipos
load_staging(data=ipos, table_name='ipos', schema='public', idx_name='ipo_id', source='database')
